# DQN for CartPole without Unity

This updated example demonstrates basic usage of `tensoraerospace.agent.dqn` on the standard `CartPole-v1` environment. The notebook has no external dependencies on Unity or unstable plugins, so it can be run in CI via `pytest --nbmake`.

In [ ]:
import gymnasium as gym
import numpy as np
import torch

from tensoraerospace.agent.dqn.model import Model, DQNAgent

In [ ]:
# Fix seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

env = gym.make('CartPole-v1')
num_actions = env.action_space.n

policy = Model(num_actions)
target_policy = Model(num_actions)
target_policy.load_state_dict(policy.state_dict())

agent = DQNAgent(
    model=policy,
    target_model=target_policy,
    env=env,
    train_nums=2_000,
    buffer_size=500,
    batch_size=32,
    target_update_iter=200,
    epsilon=0.2,
    epsilon_dacay=0.995,
    min_epsilon=0.05,
    replay_period=10,
)

print(f'Environment: {env.spec.id}')
print(f'Observation shape: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n} discrete actions')
print(f'Device: {agent.device}')

In [ ]:
# Train the agent. By default train() prints progress via tqdm.
agent.train()

In [ ]:
def evaluate(policy_model, episodes: int = 5) -> float:
    rewards = []
    for _ in range(episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        for _ in range(500):
            action, _ = policy_model.action_value(obs.reshape(1, -1))
            obs, reward, terminated, truncated, _ = env.step(int(action))
            total_reward += reward
            if terminated or truncated:
                break
        rewards.append(total_reward)
    return float(np.mean(rewards))

score = evaluate(agent.model)
print(f'Average reward over {5} episodes: {score:.2f}')

> 💡 *Smoke-test:* this notebook runs successfully with the command
> ```bash
> poetry run pytest --nbmake example/reinforcement_learning/example_dqn_unity.ipynb
> ```